# 02 - Elevation exposure (bathtub screen)
Downloads a 30 m USGS 3DEP DEM, then computes for each tract the share of area at or below +1, +2 and +3 ft (NAVD88).

> **Caveat:** a bathtub model ignores hydrologic connectivity, drainage and tides. Treat results as a screening indicator, not an inundation forecast.

In [ ]:
import sys; sys.path.insert(0, "..")
import geopandas as gpd, numpy as np, pandas as pd, py3dep, rioxarray
from rasterstats import zonal_stats
from src import config

base = gpd.read_file(config.PROCESSED / "tracts_base.geojson")
dem_path = config.RAW / "dem_30m.tif"

In [ ]:
if not dem_path.exists():
    bbox = tuple(base.total_bounds)
    dem = py3dep.get_dem(bbox, resolution=config.DEM_RESOLUTION_M, crs="EPSG:4326")
    dem.rio.to_raster(dem_path)
print("DEM ready:", dem_path)

In [ ]:
import rasterio
with rasterio.open(dem_path) as src:
    print(src.crs, src.res, src.shape, "nodata:", src.nodata)

### Zonal statistics
For each threshold, count DEM cells at or below it and divide by valid cells in the tract.

In [ ]:
from rasterstats import zonal_stats

def below_share(threshold_ft):
    thr_m = threshold_ft * config.FT_TO_M
    stats = zonal_stats(
        base, str(dem_path), all_touched=False,
        add_stats={"share": lambda x: float((x.compressed() <= thr_m).mean()) if x.count() else np.nan},
        stats=[],
    )
    return pd.Series([s["share"] for s in stats], index=base.index)

for ft in config.SLR_SCENARIOS_FT:
    base[f"below_{ft}ft"] = below_share(ft)

mean_stats = zonal_stats(base, str(dem_path), stats=["mean", "min"])
base["elev_mean_m"] = [s["mean"] for s in mean_stats]
base["low_elev_share"] = base["below_2ft"]   # used in the risk index (see config)
base[[c for c in base.columns if c.startswith("below_")]].describe()

In [ ]:
base.to_file(config.PROCESSED / "tracts_exposure.geojson", driver="GeoJSON")
base.plot(column="below_2ft", cmap="Blues", legend=True, figsize=(6,7), linewidth=0).set_title("Share of tract at/below +2 ft");